# Лабораторная работа №1. Линейная регрессия
**Предсказание RiskScore** (Kaggle MAI ML lab1)

Выполнил: Бахшалиев Максим, группа М8О-311БВ-24

Ограничение лабораторной: **только линейная регрессия** (в т.ч. для отбора/преобразования признаков). Вся нелинейность заложена в признаках; сами модели — линейные по параметрам.

**Содержание**
1. Загрузка и очистка
2. EDA
3. Нормализация (z-score, min-max)
4. Метрики (MSE, MAE, R², MAPE)
5. Класс линейной регрессии: аналитика / GD / SGD
6. Кросс-валидация (k-fold, leave-one-out)
7. Регуляризация (L1, L2, Elastic, Lp)
8. Улучшенные признаки + итоговая модель (две ветки + вес)
9. Посылка

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations_with_replacement
from pathlib import Path

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LinearRegression, SGDRegressor, Ridge, Lasso, ElasticNet
from sklearn.model_selection import cross_val_score, KFold, LeaveOneOut
from sklearn import metrics as skm
from threadpoolctl import threadpool_limits

sns.set_theme(style='whitegrid')
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

ROOT = Path.cwd()
print('Рабочая директория:', ROOT)

## 1. Загрузка и очистка данных

In [ ]:
raw = pd.read_csv(ROOT / 'train.csv')
test = pd.read_csv(ROOT / 'test.csv')
print(f'train: {raw.shape}, test: {test.shape}')

print('\nПолностью пустых строк:', raw.isna().all(axis=1).sum())
tmp = raw.dropna(how='all')
print('Дубликатов среди непустых:', tmp.duplicated().sum())
tmp = tmp.drop_duplicates()
bad_target = tmp['RiskScore'].abs() > 1e6
print(f'Заглушек в RiskScore (|y|>1e6): {bad_target.sum()} → {sorted(tmp.loc[bad_target, "RiskScore"].unique())}')
print('Пропусков в test:', test.isna().sum().sum())

In [ ]:
train = (
    raw.dropna(how='all')
       .drop_duplicates()
       .loc[lambda df: df['RiskScore'].abs() < 1e6]
       .dropna()
       .reset_index(drop=True)
)
y = train['RiskScore'].to_numpy(dtype=float)
print(f'Итоговый train: {train.shape}, целевая: mean={y.mean():.2f}, std={y.std():.2f}')

В данных были: ~530 полностью пустых строк, ~487 дубликатов, ~212 строк с заглушками ±10⁷ в `RiskScore` и блок пропусков (~450 строк). В `test` пропусков нет, поэтому такие наблюдения просто удалены. Остаётся 9332 строки.

## 2. Разведочный анализ (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
raw_score = raw.dropna(how='all').drop_duplicates()['RiskScore']
axes[0].hist(raw_score.clip(-100, 200), bins=50, color='salmon', edgecolor='k', alpha=0.8)
axes[0].set_title('RiskScore до очистки (обрезано [-100, 200])')
axes[1].hist(y, bins=50, color='steelblue', edgecolor='k', alpha=0.8)
axes[1].set_title('RiskScore после очистки')
for ax in axes:
    ax.set_xlabel('RiskScore')
plt.tight_layout()
plt.show()
print(pd.Series(y).describe())

In [ ]:
num_cols = train.select_dtypes(include=np.number).columns
corr = train[num_cols].corr()
plt.figure(figsize=(13, 10))
sns.heatmap(corr, cmap='RdBu_r', center=0, vmin=-1, vmax=1, square=True,
            cbar_kws={'shrink': 0.65})
plt.title('Матрица корреляций')
plt.show()

print('Топ корреляций с RiskScore:')
print(corr['RiskScore'].drop('RiskScore').reindex(
    corr['RiskScore'].drop('RiskScore').abs().sort_values(ascending=False).index
).head(12))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
pairs = [
    ('CreditScore', False),
    ('MonthlyIncome', True),
    ('InterestRate', False),
    ('TotalDebtToIncomeRatio', True),
    ('NetWorth', True),
    ('DebtToIncomeRatio', False),
]
for ax, (col, logx) in zip(axes.ravel(), pairs):
    ax.scatter(train[col], y, s=6, alpha=0.3, edgecolors='none')
    if logx:
        ax.set_xscale('log')
    ax.set_xlabel(col)
    ax.set_ylabel('RiskScore')
    ax.set_title(f'RiskScore vs {col}' + (' (log)' if logx else ''))
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].scatter(train['AnnualIncome'], train['MonthlyIncome'], s=6, alpha=0.35)
axes[0].set_xscale('log')
axes[0].set_title('AnnualIncome vs MonthlyIncome')
axes[1].scatter(train['BaseInterestRate'], train['InterestRate'], s=6, alpha=0.35)
axes[1].set_title('BaseInterestRate vs InterestRate')
axes[2].scatter(train['CreditScore'], train['InterestRate'], s=6, alpha=0.35)
axes[2].set_title('CreditScore vs InterestRate')
plt.tight_layout()
plt.show()

print('CreditScore == 300 / 850:', (train.CreditScore == 300).sum(), (train.CreditScore == 850).sum())
print('MonthlyIncome == 1250 / 25000:', (train.MonthlyIncome == 1250).sum(), (train.MonthlyIncome == 25000).sum())

### Выводы по EDA
- Целевая после очистки лежит примерно в [15, 98], среднее ≈ 48, std ≈ 17. Распределение **бимодально** (пики около 32 и 68) — есть два «режима» риска.
- Наиболее связанные с целью признаки: `CreditScore` (отрицательная корреляция ≈ −0.79), доходы, `InterestRate` / `BaseInterestRate`, `TotalDebtToIncomeRatio`.
- Сильная мультиколлинеарность: Age↔Experience, AnnualIncome↔MonthlyIncome, BaseInterestRate↔InterestRate, TotalAssets↔NetWorth и т.д.
- Зависимости от ключевых признаков нелинейные (S-образные / двухполосные). Много значений «прижато» к границам (CreditScore 300/850, MonthlyIncome 1250/25000) — полезны индикаторы обрезки.
- Тяжёлые хвосты у денежных величин → логарифмы / знаковые логарифмы.

## 3. Нормализация: z-score и min-max

In [ ]:
class MyStandardScaler:
    """Z-score: (x - mean) / std"""
    def fit(self, X):
        X = np.asarray(X, dtype=float)
        self.mean_ = X.mean(axis=0)
        self.std_ = X.std(axis=0)
        self.std_[self.std_ == 0] = 1.0
        return self

    def transform(self, X):
        return (np.asarray(X, dtype=float) - self.mean_) / self.std_

    def fit_transform(self, X):
        return self.fit(X).transform(X)


class MyMinMaxScaler:
    """Min-max: (x - min) / (max - min)"""
    def fit(self, X):
        X = np.asarray(X, dtype=float)
        self.min_ = X.min(axis=0)
        self.range_ = X.max(axis=0) - self.min_
        self.range_[self.range_ == 0] = 1.0
        return self

    def transform(self, X):
        return (np.asarray(X, dtype=float) - self.min_) / self.range_

    def fit_transform(self, X):
        return self.fit(X).transform(X)

In [ ]:
check_cols = ['Age', 'AnnualIncome', 'CreditScore', 'InterestRate', 'NetWorth']
A = train[check_cols].values

print('z-score совпадает с sklearn:',
      np.allclose(MyStandardScaler().fit_transform(A), StandardScaler().fit_transform(A)))
print('min-max совпадает с sklearn:',
      np.allclose(MyMinMaxScaler().fit_transform(A), MinMaxScaler().fit_transform(A)))

Z = MyStandardScaler().fit_transform(A)
print('После z-score: mean ≈', Z.mean(0).round(6), 'std ≈', Z.std(0).round(3))
M = MyMinMaxScaler().fit_transform(A)
print('После min-max: min =', M.min(0), 'max =', M.max(0))

## 4. Метрики: MSE, MAE, R², MAPE

In [ ]:
def mse(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    return np.mean((y_true - y_pred) ** 2)

def mae(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    return np.mean(np.abs(y_true - y_pred))

def r2(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return 1.0 - ss_res / ss_tot

def mape(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    denom = np.maximum(np.abs(y_true), np.finfo(float).eps)
    return np.mean(np.abs(y_true - y_pred) / denom)

In [ ]:
rng = np.random.default_rng(42)
yt = rng.normal(50, 15, 2000)
yp = yt + rng.normal(0, 4, 2000)

cmp = pd.DataFrame([
    ('MSE',  mse(yt, yp),  skm.mean_squared_error(yt, yp)),
    ('MAE',  mae(yt, yp),  skm.mean_absolute_error(yt, yp)),
    ('R2',   r2(yt, yp),   skm.r2_score(yt, yp)),
    ('MAPE', mape(yt, yp), skm.mean_absolute_percentage_error(yt, yp)),
], columns=['metric', 'моя реализация', 'sklearn'])
cmp['совпадает'] = np.isclose(cmp['моя реализация'], cmp['sklearn'], atol=1e-10)
cmp

## 5. Класс линейной регрессии (analytic / GD / SGD)

Метод обучения задаётся параметром `method`:
- `analytic` — нормальные уравнения $w = (X^\top X)^{-1} X^\top y$ (с L2-регуляризацией при необходимости);
- `gd` — полный градиентный спуск;
- `sgd` — стохастический / мини-батч с затухающим шагом.

Регуляризация (раздел 7) уже встроена: `reg ∈ {None, 'l1', 'l2', 'elastic', 'lp'}`.

In [ ]:
class MyLinearRegression:
    """
    Лосс: 1/(2n) ||y - Xw||² + α · Ω(w)
    Ω: None | L1 ||w||₁ | L2 ½||w||₂² | elastic ρ||w||₁ + ½(1-ρ)||w||₂² | Lp (1/p) Σ|w|ᵖ
    Свободный член не штрафуется.
    """
    def __init__(self, method='analytic', reg=None, alpha=0.0, l1_ratio=0.5, p=2.0,
                 lr=0.05, n_iter=3000, batch_size=32, random_state=0):
        self.method = method
        self.reg = reg
        self.alpha = alpha
        self.l1_ratio = l1_ratio
        self.p = p
        self.lr = lr
        self.n_iter = n_iter
        self.batch_size = batch_size
        self.random_state = random_state

    def _penalty_grad(self, w):
        if self.reg is None or self.alpha == 0:
            return np.zeros_like(w)
        if self.reg == 'l1':
            g = np.sign(w)
        elif self.reg == 'l2':
            g = w
        elif self.reg == 'elastic':
            g = self.l1_ratio * np.sign(w) + (1 - self.l1_ratio) * w
        elif self.reg == 'lp':
            g = np.abs(w) ** (self.p - 1) * np.sign(w)
        else:
            raise ValueError(f'unknown reg={self.reg}')
        return self.alpha * g

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        n, m = X.shape
        Xb = np.column_stack([np.ones(n), X])

        if self.method == 'analytic':
            if self.reg not in (None, 'l2'):
                raise ValueError('analytic поддерживает только reg=None или reg="l2"')
            reg_mat = np.eye(m + 1) * (self.alpha if self.reg == 'l2' else 0.0)
            reg_mat[0, 0] = 0.0
            A = Xb.T @ Xb / n + reg_mat + 1e-10 * np.eye(m + 1)
            b = Xb.T @ y / n
            self.w_ = np.linalg.solve(A, b)
        elif self.method in ('gd', 'sgd'):
            rng = np.random.default_rng(self.random_state)
            w = np.zeros(m + 1)
            bs = n if self.method == 'gd' else min(self.batch_size, n)
            for it in range(self.n_iter):
                if self.method == 'gd':
                    idx = slice(None)
                else:
                    idx = rng.integers(0, n, size=bs)
                Xi, yi = Xb[idx], y[idx]
                resid = Xi @ w - yi
                grad = Xi.T @ resid / len(yi)
                grad[1:] += self._penalty_grad(w[1:])
                step = self.lr if self.method == 'gd' else self.lr / (1 + 1e-3 * it) ** 0.75
                w -= step * grad
            self.w_ = w
        else:
            raise ValueError(self.method)

        self.intercept_ = self.w_[0]
        self.coef_ = self.w_[1:]
        return self

    def predict(self, X):
        return self.intercept_ + np.asarray(X, dtype=float) @ self.coef_

### Базовые признаки и сравнение с sklearn

In [ ]:
def train_val_split(X, y, val_frac=0.2, seed=0):
    idx = np.random.default_rng(seed).permutation(len(y))
    n_val = int(len(y) * val_frac)
    return X[idx[n_val:]], X[idx[:n_val]], y[idx[n_val:]], y[idx[:n_val]]

CAT_COLS = ['MaritalStatus', 'HomeOwnershipStatus', 'LoanPurpose', 'EmploymentStatus', 'EducationLevel']
LOG_COLS = [
    'AnnualIncome', 'LoanAmount', 'TotalAssets', 'TotalLiabilities', 'NetWorth',
    'SavingsAccountBalance', 'CheckingAccountBalance', 'MonthlyDebtPayments',
    'MonthlyLoanPayment', 'MonthlyIncome', 'TotalDebtToIncomeRatio'
]

def make_base_features(df):
    d = df.drop(columns=[c for c in ['ID', 'RiskScore', 'ApplicationDate'] if c in df], errors='ignore').copy()
    for c in LOG_COLS:
        if c in d.columns:
            d[f'log_{c}'] = np.log1p(d[c].clip(lower=0))
    d['cs_low']  = (d['CreditScore'] <= 300).astype(float)
    d['cs_high'] = (d['CreditScore'] >= 850).astype(float)
    d['mi_low']  = (d['MonthlyIncome'] <= 1250).astype(float)
    d['mi_high'] = (d['MonthlyIncome'] >= 25000).astype(float)
    return pd.get_dummies(d, columns=[c for c in CAT_COLS if c in d.columns],
                          drop_first=True, dtype=float)

F_base = make_base_features(train)
print('Базовая матрица признаков:', F_base.shape)

Xtr, Xval, ytr, yval = train_val_split(F_base.values, y, val_frac=0.2, seed=0)
scaler_z = MyStandardScaler().fit(Xtr)
Xtr_z, Xval_z = scaler_z.transform(Xtr), scaler_z.transform(Xval)

In [ ]:
models = {
    'моя analytic': MyLinearRegression('analytic'),
    'моя GD':       MyLinearRegression('gd', lr=0.08, n_iter=4000),
    'моя SGD':      MyLinearRegression('sgd', lr=0.02, n_iter=40000, batch_size=16),
    'sklearn LR':   LinearRegression(),
    'sklearn SGD':  SGDRegressor(penalty=None, max_iter=100, tol=None, random_state=0),
}

rows = []
for name, model in models.items():
    model.fit(Xtr_z, ytr)
    pred = model.predict(Xval_z)
    rows.append((name, mse(yval, pred), mae(yval, pred), r2(yval, pred), mape(yval, pred)))

pd.DataFrame(rows, columns=['модель', 'MSE', 'MAE', 'R2', 'MAPE']).set_index('модель')

In [ ]:
my_a = MyLinearRegression('analytic').fit(Xtr_z, ytr)
sk_a = LinearRegression().fit(Xtr_z, ytr)
print('max |w_моя − w_sklearn| (analytic):', np.abs(my_a.coef_ - sk_a.coef_).max())

mm = MyMinMaxScaler().fit(Xtr)
Xtr_m, Xval_m = mm.transform(Xtr), mm.transform(Xval)
print('\nMSE (analytic): z-score = %.3f | min-max = %.3f' % (
    mse(yval, MyLinearRegression('analytic').fit(Xtr_z, ytr).predict(Xval_z)),
    mse(yval, MyLinearRegression('analytic').fit(Xtr_m, ytr).predict(Xval_m))))

Аналитическое решение совпадает со sklearn (разница коэффициентов ~1e-5 из-за коллинеарности). GD сходится к тому же, SGD чуть шумнее. Базовая линейная модель даёт MSE ≈ 33–34 — выше порога. Нужны более выразительные признаки.

## 6. Кросс-валидация: k-fold и leave-one-out

In [ ]:
def kfold_cv(model, X, y, k=5, metric=mse, seed=0):
    X, y = np.asarray(X, float), np.asarray(y, float)
    idx = np.random.default_rng(seed).permutation(len(y))
    folds = np.array_split(idx, k)
    scores = []
    for i in range(k):
        va = folds[i]
        tr = np.concatenate([folds[j] for j in range(k) if j != i])
        model.fit(X[tr], y[tr])
        scores.append(metric(y[va], model.predict(X[va])))
    return np.asarray(scores)

def leave_one_out(model, X, y, max_folds=None, seed=0):
    X, y = np.asarray(X, float), np.asarray(y, float)
    n = len(y)
    held = np.arange(n) if max_folds is None else np.random.default_rng(seed).choice(n, max_folds, replace=False)
    errs = []
    for i in held:
        mask = np.ones(n, dtype=bool)
        mask[i] = False
        model.fit(X[mask], y[mask])
        errs.append((y[i] - model.predict(X[i:i+1])[0]) ** 2)
    return float(np.mean(errs))

In [ ]:
Xz_all = MyStandardScaler().fit_transform(F_base.values)

for k in (5, 10):
    mine = kfold_cv(MyLinearRegression('analytic'), Xz_all, y, k=k).mean()
    sk = -cross_val_score(LinearRegression(), Xz_all, y,
                          cv=KFold(k, shuffle=True, random_state=0),
                          scoring='neg_mean_squared_error').mean()
    print(f'{k}-fold CV MSE: моя = {mine:.3f} | sklearn = {sk:.3f}')

print('\nLOO (1000 случайных объектов):',
      round(leave_one_out(MyLinearRegression('analytic'), Xz_all, y, max_folds=1000), 3))

## 7. Регуляризация (L1 / L2 / Elastic / Lp)

In [ ]:
alpha = 0.05
n_tr = len(ytr)
rows = []

def add_row(name, model):
    model.fit(Xtr_z, ytr)
    p = model.predict(Xval_z)
    rows.append((
        name,
        mse(yval, p),
        mae(yval, p),
        r2(yval, p),
        np.abs(model.coef_).sum(),
        (np.abs(model.coef_) < 1e-2).sum()
    ))

add_row('без регуляризации', MyLinearRegression('analytic'))
add_row('L2 analytic',       MyLinearRegression('analytic', reg='l2', alpha=alpha))
add_row('L2 GD',             MyLinearRegression('gd', reg='l2', alpha=alpha, lr=0.08, n_iter=4000))
add_row('L2 Ridge (sklearn)', Ridge(alpha=alpha * n_tr))
add_row('L1 GD',             MyLinearRegression('gd', reg='l1', alpha=alpha, lr=0.08, n_iter=4000))
add_row('L1 Lasso (sklearn)', Lasso(alpha=alpha))
add_row('Elastic GD (ρ=0.5)', MyLinearRegression('gd', reg='elastic', alpha=alpha, l1_ratio=0.5, lr=0.08, n_iter=4000))
add_row('ElasticNet sklearn', ElasticNet(alpha=alpha, l1_ratio=0.5))
add_row('Lp GD (p=1.5)',     MyLinearRegression('gd', reg='lp', p=1.5, alpha=alpha, lr=0.08, n_iter=4000))
add_row('Lp GD (p=3)',       MyLinearRegression('gd', reg='lp', p=3.0, alpha=alpha, lr=0.08, n_iter=4000))

pd.DataFrame(rows, columns=['модель', 'MSE', 'MAE', 'R2', 'Σ|w|', 'число |w|<0.01']).set_index('модель')

На базовых признаках регуляризация почти не улучшает MSE — модель недообучена, а не переобучена. Она понадобится при большом числе признаков в следующем разделе.

## 8. Улучшенные признаки и итоговая модель

Из EDA видно, что `RiskScore` бимодален. Поэтому используем **две линейные ветки** (низкий / высокий риск) и третью линейную модель, которая предсказывает вес верхней ветки. Все три модели — обычная линейная регрессия.

Основные признаки — пороговые индикаторы (ступеньки) по ключевым числовым переменным + несколько бинарных флагов. Дополнительно — отношения и знаковые логарифмы.

In [ ]:
# ---------- пороговые признаки ----------
MAIN_THRESH = {
    'CreditScore':              [580, 620, 660, 700, 740, 780],
    'DebtToIncomeRatio':        [0.15, 0.25, 0.35, 0.45, 0.55],
    'CreditCardUtilizationRate':[0.15, 0.30, 0.45, 0.60, 0.75],
    'LengthOfCreditHistory':    [2, 5, 8, 12, 18],
    'AnnualIncome':             [25_000, 40_000, 60_000, 90_000, 130_000],
    'NetWorth':                 [20_000, 60_000, 150_000, 350_000, 700_000],
}

EXTRA_THRESH = {
    'CreditScore':              [500, 550, 800, 850],
    'MonthlyIncome':            np.array([12, 18, 30, 50, 80, 120, 180, 250]) * 1000 / 12,
    'DebtToIncomeRatio':        [0.40, 0.65],
    'TotalDebtToIncomeRatio':   [0.08, 0.18, 0.28, 0.40, 0.55, 0.9, 1.5],
    'NetWorth':                 [5_000, 15_000, 100_000, 500_000, 1_200_000],
    'Age':                      [22, 28, 34, 40, 46, 52, 58, 64],
    'CreditCardUtilizationRate':[0.08, 0.25, 0.55, 0.85],
    'LengthOfCreditHistory':    [4, 10, 15, 22],
}


def threshold_feats(frame, grids):
    out = {}
    for name, thresholds in grids.items():
        if name not in frame.columns:
            continue
        for t in thresholds:
            out[f'{name}>={t}'] = frame[name].ge(t).astype(float)
    return out


def main_features(frame):
    feats = threshold_feats(frame, MAIN_THRESH)
    feats['Bankruptcy'] = frame['BankruptcyHistory'].astype(float)
    feats['PrevDefaults'] = frame['PreviousLoanDefaults'].astype(float)
    feats['NotEmployed'] = (frame['EmploymentStatus'] != 'Employed').astype(float)
    return pd.DataFrame(feats, index=frame.index)


def financial_ratios(frame):
    inc = frame['AnnualIncome'].replace(0, np.nan)
    mon = frame['MonthlyIncome'].replace(0, np.nan)
    return pd.DataFrame({
        'IncomePerDep':      frame['AnnualIncome'] / (frame['NumberOfDependents'] + 1),
        'DebtBurden':        frame['MonthlyDebtPayments'] / mon,
        'LoanToIncome':      frame['LoanAmount'] / inc,
        'PaymentToIncome':   frame['MonthlyLoanPayment'] / mon,
        'AssetsToIncome':    frame['TotalAssets'] / inc,
        'LiabToAssets':      frame['TotalLiabilities'] / frame['TotalAssets'].replace(0, np.nan),
        'NetWorthToIncome':  frame['NetWorth'] / inc,
        'SavingsToIncome':   frame['SavingsAccountBalance'] / inc,
        'CheckingToIncome':  frame['CheckingAccountBalance'] / inc,
        'LiqAssetsToDebt':   (frame['SavingsAccountBalance'] + frame['CheckingAccountBalance'])
                             / frame['TotalLiabilities'].replace(0, np.nan),
        'HistPerAge':        frame['LengthOfCreditHistory'] / frame['Age'].replace(0, np.nan),
        'ExpPerAge':         frame['Experience'] / frame['Age'].replace(0, np.nan),
        'LinesPerHist':      frame['NumberOfOpenCreditLines'] / frame['LengthOfCreditHistory'].replace(0, np.nan),
        'InqPerLine':        frame['NumberOfCreditInquiries'] / (frame['NumberOfOpenCreditLines'] + 1),
        'RateRelative':      frame['InterestRate'] / frame['BaseInterestRate'].replace(0, np.nan),
        'ScorePerInq':       frame['CreditScore'] / (frame['NumberOfCreditInquiries'] + 1),
    })


def extra_features(frame):
    steps = pd.DataFrame(threshold_feats(frame, EXTRA_THRESH), index=frame.index)
    ratios = financial_ratios(frame)
    signed_log = np.sign(ratios) * np.log1p(ratios.abs())
    out = pd.concat([steps, signed_log], axis=1)
    return out.replace([np.inf, -np.inf], np.nan).fillna(0.0)

In [ ]:
main_tr = main_features(train)
main_te = main_features(test)
print('Основных пороговых признаков:', main_tr.shape[1])


def assign_branches(steps, target, max_iter=40):
    """Итеративное разбиение на нижнюю/верхнюю ветку по близости прогноза."""
    upper = target > np.median(target)
    for _ in range(max_iter):
        low_m  = LinearRegression().fit(steps.loc[~upper], target[~upper])
        high_m = LinearRegression().fit(steps.loc[upper],  target[upper])
        low_p  = low_m.predict(steps)
        high_p = high_m.predict(steps)
        new_upper = np.abs(target - high_p) < np.abs(target - low_p)
        if np.array_equal(upper, new_upper):
            break
        upper = new_upper
    # гарантируем, что «верхняя» ветка имеет больший средний прогноз
    if high_p.mean() < low_p.mean():
        upper = ~upper
    return upper


with threadpool_limits(limits=2):
    is_upper = assign_branches(main_tr, y)
print(f'Нижняя ветка: {(~is_upper).sum()}, верхняя: {is_upper.sum()}')

In [ ]:
# ---------- признаки для веса ветки ----------
EDU_ORDER = {'High School': 0, 'Associate': 1, 'Bachelor': 2, 'Master': 3, 'Doctorate': 4}
CAT_FOR_INDEX = ['MaritalStatus', 'HomeOwnershipStatus', 'LoanPurpose', 'EmploymentStatus']


def index_features(frame):
    num = frame.select_dtypes(include=np.number).drop(
        columns=['ID', 'AnnualIncome', 'BaseInterestRate'], errors='ignore'
    ).astype(float)
    num['AgeSq'] = frame['Age'].astype(float) ** 2
    num['AppMonth'] = pd.to_datetime(frame['ApplicationDate']).dt.month
    num['EduOrd'] = frame['EducationLevel'].map(EDU_ORDER).astype(float)
    cats = pd.get_dummies(frame[CAT_FOR_INDEX], drop_first=True, dtype=float)
    return pd.concat([num, cats], axis=1)


idx_tr = index_features(train)
idx_te = index_features(test).reindex(columns=idx_tr.columns, fill_value=0.0)

idx_scaler = StandardScaler().fit(idx_tr)
Z_tr = idx_scaler.transform(idx_tr)
Z_te = idx_scaler.transform(idx_te)

# несколько проходов, уточняющих границу
mask = np.ones(len(train), dtype=bool)
with threadpool_limits(limits=2):
    for _ in range(6):
        gate = LinearRegression().fit(Z_tr[mask], is_upper[mask].astype(float))
        raw_p = gate.predict(Z_tr)
        mask = (raw_p > 0) & (raw_p < 1)

p_test = np.clip(gate.predict(Z_te), 0.0, 1.0)

In [ ]:
# ---------- обучение веток с лёгкой сглаживающей регуляризацией ----------
extra_tr = extra_features(train)
extra_te = extra_features(test).clip(extra_tr.min(), extra_tr.max(), axis='columns')

extra_sc = StandardScaler().fit(extra_tr)
phi_tr = np.column_stack([np.ones(len(train)), main_tr.values, extra_sc.transform(extra_tr)])
phi_te = np.column_stack([np.ones(len(test)),  main_te.values, extra_sc.transform(extra_te)])

n_feat = phi_tr.shape[1]
unit = np.eye(n_feat)

# штрафы на разности соседних порогов одного признака + L2 на extra-признаки
diff_rows = []
start = 1
for thresholds in MAIN_THRESH.values():
    k = len(thresholds)
    if k > 1:
        diff_rows.append(unit[start:start+k-1] - unit[start+1:start+k])
    start += k

n_main = main_tr.shape[1]
penalties = np.vstack([
    np.sqrt(len(train) * 80 / 6000) * np.vstack(diff_rows),
    np.sqrt(len(train) * 8000 / 6000) * unit[1 + n_main:],
    1e-4 * unit,
])
zeros = np.zeros(len(penalties))

with threadpool_limits(limits=2):
    branch_models = [
        LinearRegression(fit_intercept=False).fit(
            np.vstack([phi_tr[mask], penalties]),
            np.concatenate([y[mask], zeros])
        )
        for mask in (~is_upper, is_upper)
    ]

print(f'Признаков на ветку (включая intercept): {n_feat}')

In [ ]:
# ---------- прогноз ----------
low_pred  = branch_models[0].predict(phi_te)
high_pred = branch_models[1].predict(phi_te)
final_pred = (1 - p_test) * low_pred + p_test * high_pred

submission = pd.DataFrame({'ID': test['ID'], 'RiskScore': final_pred})
assert len(submission) == len(test)
assert submission['ID'].is_unique
assert np.isfinite(submission['RiskScore']).all()

out_path = ROOT / 'submission.csv'
submission.to_csv(out_path, index=False)
print(f'Сохранено {len(submission)} прогнозов → {out_path.name}')
print(submission['RiskScore'].describe())
display(submission.head())

### Краткая проверка на hold-out (для самоконтроля)

In [ ]:
# быстрая оценка качества на 20% hold-out (не используется для финальной посылки)
rng = np.random.default_rng(1)
perm = rng.permutation(len(train))
val_idx, tr_idx = perm[:len(train)//5], perm[len(train)//5:]

main_h = main_features(train.iloc[tr_idx])
y_h = y[tr_idx]
upper_h = assign_branches(main_h, y_h)

extra_h = extra_features(train.iloc[tr_idx])
extra_sc_h = StandardScaler().fit(extra_h)
phi_h = np.column_stack([np.ones(len(tr_idx)), main_h.values, extra_sc_h.transform(extra_h)])

n_f = phi_h.shape[1]
unit_h = np.eye(n_f)
diff_h = []
st = 1
for th in MAIN_THRESH.values():
    k = len(th)
    if k > 1:
        diff_h.append(unit_h[st:st+k-1] - unit_h[st+1:st+k])
    st += k
pens = np.vstack([
    np.sqrt(len(tr_idx) * 80 / 6000) * np.vstack(diff_h),
    np.sqrt(len(tr_idx) * 8000 / 6000) * unit_h[1 + main_h.shape[1]:],
    1e-4 * unit_h,
])
zeros_h = np.zeros(len(pens))

models_h = [
    LinearRegression(fit_intercept=False).fit(
        np.vstack([phi_h[m], pens]), np.concatenate([y_h[m], zeros_h])
    )
    for m in (~upper_h, upper_h)
]

main_v = main_features(train.iloc[val_idx])
extra_v = extra_features(train.iloc[val_idx]).clip(extra_h.min(), extra_h.max(), axis='columns')
phi_v = np.column_stack([np.ones(len(val_idx)), main_v.values, extra_sc_h.transform(extra_v)])

idx_h = index_features(train.iloc[tr_idx])
idx_v = index_features(train.iloc[val_idx]).reindex(columns=idx_h.columns, fill_value=0.0)
sc_h = StandardScaler().fit(idx_h)
gate_h = LinearRegression().fit(sc_h.transform(idx_h), upper_h.astype(float))
p_v = np.clip(gate_h.predict(sc_h.transform(idx_v)), 0, 1)

pred_v = (1 - p_v) * models_h[0].predict(phi_v) + p_v * models_h[1].predict(phi_v)
print(f'Hold-out MSE ≈ {mse(y[val_idx], pred_v):.3f}')
print(f'Hold-out MAE ≈ {mae(y[val_idx], pred_v):.3f}')
print(f'Hold-out R²  ≈ {r2(y[val_idx], pred_v):.3f}')

## Итог

- Все обязательные пункты лабораторной реализованы вручную и сверены со sklearn.
- Базовая линейная модель на «сырых» признаках даёт MSE ≈ 33–34.
- Итоговая модель — смесь двух линейных регрессий с линейным гейтом. Она остаётся полностью в рамках линейной регрессии, но учитывает бимодальность целевой переменной через пороговые и отношение-признаки.
- Ожидаемый публичный score на Kaggle — около 18.5–19.0 (ниже порога 19 и значительно ниже 25).

Файл `submission.csv` готов к загрузке.